# EnvelopeManager: Coordinating Multiple Envelopes

The `EnvelopeManager` orchestrates envelope creation, contribution scheduling, and manages collections of envelopes. It implements non-overlapping contribution periods for recurring bills, ensuring you contribute to only one envelope per bill at a time.

**Topics covered:**
- Creating envelopes from bill instances
- Setting non-overlapping contribution periods
- Batch operations and duplicate prevention


In [ ]:
from datetime import date
from decimal import Decimal
from sinkingfund.managers import EnvelopeManager
from sinkingfund.models import BillInstance


## Creating Envelopes from Bill Instances

The `create_envelopes()` method creates envelope objects from a list of bill instances. These envelopes are ready to be added to the manager.


In [ ]:
# Create bill instances for a recurring bill.
instances = [
    BillInstance(
        bill_id="electric",
        service="Electric Bill",
        due_date=date(2025, 3, 15),
        amount_due=Decimal("150.00")
    ),
    BillInstance(
        bill_id="electric",
        service="Electric Bill",
        due_date=date(2025, 4, 15),
        amount_due=Decimal("145.00")
    ),
    BillInstance(
        bill_id="electric",
        service="Electric Bill",
        due_date=date(2025, 5, 15),
        amount_due=Decimal("155.00")
    ),
]

# Create EnvelopeManager.
manager = EnvelopeManager()

# Create envelopes from instances.
envelopes = manager.create_envelopes(instances)
manager.add_envelopes(envelopes)

print(f"Created {len(manager.envelopes)} envelopes")
for envelope in manager.envelopes:
    print(f"  {envelope.bill_instance.service} due {envelope.bill_instance.due_date}")


## Setting Non-Overlapping Contribution Periods

The `set_contrib_dates()` method assigns contribution windows to all envelopes. For recurring bills, it ensures contribution periods don't overlap - you contribute to one envelope at a time.


In [ ]:
# Set contribution dates with bi-weekly intervals.
# The manager automatically creates non-overlapping periods for recurring bills.
manager.set_contrib_dates(
    start_contrib_date=date(2025, 1, 1),
    contrib_interval=14  # Bi-weekly
)

print("=== Contribution Periods (Non-Overlapping) ===")
for envelope in manager.envelopes:
    print(f"{envelope.bill_instance.service} due {envelope.bill_instance.due_date}:")
    print(f"  Contribution window: {envelope.start_contrib_date} to {envelope.end_contrib_date}")
    print()


Notice how each envelope gets a non-overlapping contribution period. The first envelope ends when the second begins, ensuring you're only contributing to one electric bill envelope at a time.


## Batch Operations

EnvelopeManager supports adding multiple envelopes at once through `add_envelopes()`. It validates for duplicates based on bill_id and due_date combinations.


In [ ]:
# Create additional bill instances from different bills.
additional_instances = [
    BillInstance(
        bill_id="prop_tax",
        service="Property Tax",
        due_date=date(2025, 11, 1),
        amount_due=Decimal("3600.00")
    ),
    BillInstance(
        bill_id="insurance",
        service="Car Insurance",
        due_date=date(2025, 6, 1),
        amount_due=Decimal("750.00")
    ),
]

# Create and add envelopes in batch.
additional_envelopes = manager.create_envelopes(additional_instances)
manager.add_envelopes(additional_envelopes)

print(f"Total envelopes managed: {len(manager.envelopes)}")
print("\nAll envelopes:")
for envelope in manager.envelopes:
    print(f"  {envelope.bill_instance.bill_id} - {envelope.bill_instance.service} due {envelope.bill_instance.due_date}")


### Duplicate Prevention

The manager prevents duplicate envelopes (same bill_id and due_date):


In [ ]:
# Try to add a duplicate envelope (same bill_id and due_date).
duplicate_instance = BillInstance(
    bill_id="electric",
    service="Electric Bill",
    due_date=date(2025, 3, 15),  # Same as first envelope
    amount_due=Decimal("150.00")
)

try:
    duplicate_envelope = manager.create_envelopes([duplicate_instance])
    manager.add_envelopes(duplicate_envelope)
    print("Duplicate added (shouldn't happen)")
except ValueError as e:
    print(f"Duplicate prevented: {e}")


## Summary

**Key Points:**

1. **Envelope Creation**: `create_envelopes()` converts bill instances to envelope objects
2. **Non-Overlapping Periods**: `set_contrib_dates()` ensures recurring bills have sequential contribution windows
3. **Batch Operations**: Use `add_envelopes()` to add multiple envelopes at once
4. **Duplicate Prevention**: Manager validates uniqueness by (bill_id, due_date) combination

**Next Steps:**
- Learn about allocation strategies in the AllocationManager notebook
- See how envelopes work with contribution schedules
